<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_IRCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# In[1]:
# Cell 1: Setup and Gross Positions (Corresponds to Step 3 in the report)
# ----------------------------------------------------------------------------
# This cell loads the initial portfolio data into a pandas DataFrame and defines
# the key regulatory parameters. The output displays the starting gross curvature
# risk positions for the portfolio.

import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# CVR+ represents the net loss from an upward shock.
# CVR- represents the net loss from a downward shock.
portfolio_data = [
    {'Bucket': 'EUR', 'Curve': 'OIS Curve', 'CVR+': 24847, 'CVR-': 26813},
    {'Bucket': 'EUR', 'Curve': 'OIS Curve', 'CVR+': 13059, 'CVR-': 14313},
    {'Bucket': 'EUR', 'Curve': 'Implied Curve', 'CVR+': 118705, 'CVR-': 143728},
    {'Bucket': 'EUR', 'Curve': 'Implied Curve', 'CVR+': -17172, 'CVR-': -18164},
    {'Bucket': 'USD', 'Curve': 'OIS Curve', 'CVR+': 3375, 'CVR-': -2736},
    {'Bucket': 'USD', 'Curve': 'OIS Curve', 'CVR+': 14325, 'CVR-': 15634},
    {'Bucket': 'USD', 'Curve': 'Implied Curve', 'CVR+': -38362, 'CVR-': 105626},
    {'Bucket': 'USD', 'Curve': 'Implied Curve', 'CVR+': -26976, 'CVR-': -37288}
]

# --- Regulatory Parameters (Medium Scenario) ---
# For GIRR Curvature, the correlation is the square of the delta correlation (50%).
GAMMA_BC_MEDIUM = 0.50 ** 2

# Create the initial DataFrame
df_gross = pd.DataFrame(portfolio_data)

print("--- Step 3: Establish Gross Positions ---")
print("The calculation begins with the gross curvature risk positions for each instrument.")
print("\n" + df_gross.to_string(index=False))

--- Step 3: Establish Gross Positions ---
The calculation begins with the gross curvature risk positions for each instrument.

Bucket         Curve   CVR+   CVR-
   EUR     OIS Curve  24847  26813
   EUR     OIS Curve  13059  14313
   EUR Implied Curve 118705 143728
   EUR Implied Curve -17172 -18164
   USD     OIS Curve   3375  -2736
   USD     OIS Curve  14325  15634
   USD Implied Curve -38362 105626
   USD Implied Curve -26976 -37288


In [8]:
# In[2]:
# Cell 2: Net Positions (Corresponds to Step 4 in the report)
# ----------------------------------------------------------------------------
# As per Article 325l, all interest rate curves within a single currency are
# treated as a single risk factor. Therefore, we group by 'Bucket' and sum the
# CVR+ and CVR- values to get the net positions.

df_net = df_gross.groupby('Bucket')[['CVR+', 'CVR-']].sum().reset_index()

print("--- Step 4: Calculate Net Positions ---")
print("Gross positions are summed within each bucket (currency) to get the net risk positions.")
print("\n" + df_net.to_string(index=False))

--- Step 4: Calculate Net Positions ---
Gross positions are summed within each bucket (currency) to get the net risk positions.

Bucket   CVR+   CVR-
   EUR 139439 166690
   USD -47638  81236


In [9]:
# In[3]:
# Cell 3: Bucket-Level Capital (Corresponds to Step 8 in the report)
# ----------------------------------------------------------------------------
# Here, we calculate the capital for each bucket (Kb). This involves finding the
# capital for the upward (Kb+) and downward (Kb-) scenarios and selecting the larger
# of the two. The formula simplifies because there is only one risk factor per bucket.

df_capital = df_net.copy()

# Calculate capital for each scenario (floored at zero)
df_capital['K_b+'] = df_capital['CVR+'].apply(lambda x: max(x, 0))
df_capital['K_b-'] = df_capital['CVR-'].apply(lambda x: max(x, 0))

# Determine the final bucket capital and the selected scenario
df_capital['K_b'] = df_capital[['K_b+', 'K_b-']].max(axis=1)
df_capital['Selected Scenario'] = np.where(df_capital['K_b+'] > df_capital['K_b-'], 'Upward', 'Downward')
# Handle the edge case where K_b+ == K_b-
df_capital.loc[df_capital['K_b+'] == df_capital['K_b-'], 'Selected Scenario'] = \
    np.where(df_capital.loc[df_capital['K_b+'] == df_capital['K_b-'], 'CVR+'] > \
             df_capital.loc[df_capital['K_b+'] == df_capital['K_b-'], 'CVR-'], 'Upward', 'Downward')


print("--- Step 8: Calculate Bucket-Level Capital (K_b) ---")
print("Capital is calculated for up/down scenarios, and the maximum is selected for each bucket.")
print("\n" + df_capital[['Bucket', 'K_b+', 'K_b-', 'K_b', 'Selected Scenario']].to_string(index=False))

--- Step 8: Calculate Bucket-Level Capital (K_b) ---
Capital is calculated for up/down scenarios, and the maximum is selected for each bucket.

Bucket   K_b+   K_b-    K_b Selected Scenario
   EUR 139439 166690 166690          Downward
   USD      0  81236  81236          Downward


In [10]:
# In[4]:
# Cell 4: Bucket Sums (Corresponds to Step 9 in the report)
# ----------------------------------------------------------------------------
# The bucket sum (Sb) is the CVR value from the scenario that was selected in the
# previous step. This value is used for the cross-bucket aggregation and is not
# floored at zero, allowing for hedging benefits.

df_sum = df_capital.copy()

# Determine Sb based on the selected scenario
df_sum['S_b'] = np.where(df_sum['Selected Scenario'] == 'Upward', df_sum['CVR+'], df_sum['CVR-'])

print("--- Step 9: Determine Bucket Sums (S_b) ---")
print("The Bucket Sum (S_b) is the CVR value from the selected scenario for each bucket.")
print("\n" + df_sum[['Bucket', 'Selected Scenario', 'S_b']].to_string(index=False))

--- Step 9: Determine Bucket Sums (S_b) ---
The Bucket Sum (S_b) is the CVR value from the selected scenario for each bucket.

Bucket Selected Scenario    S_b
   EUR          Downward 166690
   USD          Downward  81236


In [11]:
# In[5]:
# Cell 5: Cross-Bucket Capital - Medium Scenario (Corresponds to Step 10)
# ----------------------------------------------------------------------------
# We now aggregate the results for the Medium Scenario to get the final capital charge.
# This involves the bucket capitals (K_b), bucket sums (S_b), and the cross-bucket
# correlation, while applying the safeguard function.

# Extract necessary values for the calculation
k_eur = df_sum[df_sum['Bucket'] == 'EUR']['K_b'].iloc[0]
k_usd = df_sum[df_sum['Bucket'] == 'USD']['K_b'].iloc[0]
s_eur = df_sum[df_sum['Bucket'] == 'EUR']['S_b'].iloc[0]
s_usd = df_sum[df_sum['Bucket'] == 'USD']['S_b'].iloc[0]

# Safeguard Function (psi)
psi = 0 if s_eur < 0 and s_usd < 0 else 1

# Aggregation formula
sum_k_sq = k_eur**2 + k_usd**2
cross_term = 2 * GAMMA_BC_MEDIUM * s_eur * s_usd * psi

rccr_medium = np.sqrt(max(0, sum_k_sq + cross_term))

print("--- Step 10: Cross-Bucket Capital (Medium Scenario) ---")
print("Aggregating bucket results using the Medium Scenario correlation (25%).\n")
print(f"Recap of Inputs:")
print(f"  - K_EUR: {k_eur:,.0f}")
print(f"  - K_USD: {k_usd:,.0f}")
print(f"  - S_EUR: {s_eur:,.0f}")
print(f"  - S_USD: {s_usd:,.0f}")
print(f"  - Gamma: {GAMMA_BC_MEDIUM:.2%}")
print(f"  - Psi:   {psi}\n")

print("Calculation:")
print(f"  - Sum of K_b^2: {sum_k_sq:,.0f}")
print(f"  - Cross Term:   {cross_term:,.0f}")
print(f"  - Total under sqrt: {sum_k_sq + cross_term:,.0f}")
print("-------------------------------------------------")
print(f"Medium Scenario Capital: {rccr_medium:,.0f}")
print("-------------------------------------------------")

--- Step 10: Cross-Bucket Capital (Medium Scenario) ---
Aggregating bucket results using the Medium Scenario correlation (25%).

Recap of Inputs:
  - K_EUR: 166,690
  - K_USD: 81,236
  - S_EUR: 166,690
  - S_USD: 81,236
  - Gamma: 25.00%
  - Psi:   1

Calculation:
  - Sum of K_b^2: 34,384,843,796
  - Cross Term:   6,770,614,420
  - Total under sqrt: 41,155,458,216
-------------------------------------------------
Medium Scenario Capital: 202,868
-------------------------------------------------


In [12]:
# In[6]:
# Cell 6: Correlation Scenarios & Final Capital (Corresponds to Step 11)
# ----------------------------------------------------------------------------
# The final capital charge is the maximum of three scenarios, calculated by
# adjusting the correlation parameter for High and Low stress scenarios as per
# Article 325h.

def calculate_capital_for_scenario(gamma):
    """Helper function to recalculate capital based on a given gamma."""
    current_psi = 0 if s_eur < 0 and s_usd < 0 else 1
    current_sum_k_sq = k_eur**2 + k_usd**2
    current_cross_term = 2 * gamma * s_eur * s_usd * current_psi
    return np.sqrt(max(0, current_sum_k_sq + current_cross_term))

# High Scenario Correlation
gamma_high = min(GAMMA_BC_MEDIUM * 1.25, 1.0)
rccr_high = calculate_capital_for_scenario(gamma_high)

# Low Scenario Correlation
gamma_low = max(2 * GAMMA_BC_MEDIUM - 1.0, 0.75 * GAMMA_BC_MEDIUM)
rccr_low = calculate_capital_for_scenario(gamma_low)

# Final Capital
final_capital = max(rccr_medium, rccr_high, rccr_low)

# Create summary DataFrame
scenario_data = {
    'Scenario': ['Medium', 'High', 'Low'],
    'Cross-Bucket Correlation': [f"{GAMMA_BC_MEDIUM:.2%}", f"{gamma_high:.2%}", f"{gamma_low:.2%}"],
    'Calculated Capital': [rccr_medium, rccr_high, rccr_low]
}
df_scenarios = pd.DataFrame(scenario_data)

# Find the winning scenario BEFORE formatting the capital column as a string
winning_scenario = df_scenarios.loc[df_scenarios['Calculated Capital'].idxmax(), 'Scenario']

# Now, format the capital column for display purposes
df_scenarios['Calculated Capital'] = df_scenarios['Calculated Capital'].map('{:,.0f}'.format)


print("--- Step 11: Correlation Scenarios & Final Capital ---")
print("The aggregation is re-run for High and Low correlation scenarios.\n")
print(df_scenarios.to_string(index=False))

print("\n-------------------------------------------------")
print(f"Final GIRR Curvature Capital Requirement: {final_capital:,.0f}")
# This line will now execute correctly
print(f"(Driven by the '{winning_scenario}' scenario)")
print("-------------------------------------------------")



--- Step 11: Correlation Scenarios & Final Capital ---
The aggregation is re-run for High and Low correlation scenarios.

Scenario Cross-Bucket Correlation Calculated Capital
  Medium                   25.00%            202,868
    High                   31.25%            206,998
     Low                   18.75%            198,652

-------------------------------------------------
Final GIRR Curvature Capital Requirement: 206,998
(Driven by the 'High' scenario)
-------------------------------------------------
